In [0]:
# created connection for earthquake data of http type

spark.sql("DROP connection if exists api_conn_earthquake")
spark.sql("""
create connection if not exists api_conn_earthquake
  type HTTP
  options(
    host = 'https://earthquake.usgs.gov',
    port = '443',
    base_path = '/earthquakes/feed/v1.0/',
    bearer_token = 'na '
  )
""")


In [0]:
#getting the connection details(base_url) from workspace client

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
conn=w.connections.get('api_conn_earthquake')
print(conn)
base_url=f'{conn.options['host']}{conn.options['base_path']}'
# display(base_url)

In [0]:
# %sql
# use catalog  dev_catalog;
# use schema bronze;
# create volume if not exists earthquake_volume

In [0]:
#parameterizing the catalog_name using dbutils.widgets

dbutils.widgets.text('catalog_name','dev_catalog','dev_catalog')
catalog_name=dbutils.widgets.get('catalog_name')
print(catalog_name)

In [0]:
# creating a volume with in the catalog and schema

spark.sql(f'use catalog {catalog_name}')
spark.sql("use schema bronze")
spark.sql( "create volume if not exists earthquake_volume")
          

In [0]:
# calling the API to get the data and dump that into the created volume as json file with timestamp of the current date

import requests
import json
import datetime

url = f"{base_url}/summary/all_day.geojson"
response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"Error {response.status_code} - {response.text}")
data = response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/earthquake_volume/earthquake_data_{current_date}.json",
    json.dumps(data),
    overwrite=True,
)